# No.5 圧縮センシング — TV（Total Variation）版

## Curveletについて

MATLAB版 `cs_curvelet.m` は Gabriel Peyre 氏の独自ツールボックスを使用しており、
**Python対応のpipパッケージが存在しません。**

そこで同じ「アンダーサンプリングしたk空間からの画像再構成」を
**Total Variation（TV）正則化**で実現します。
TV-CSはCurvelet CSと同じ圧縮センシングの枠組みであり、実際のMRI研究でも広く使われています。

**アルゴリズム（FISTA的な繰り返し）:**
1. k空間でデータ整合性を強制（観測値で置換）
2. TV収縮（Chambolle法）でスパースな解を求める
3. 繰り返す

In [ ]:
import numpy as np
import skimage.io
import skimage.metrics
import skimage.restoration
import matplotlib.pyplot as plt
import japanize_matplotlib


In [ ]:
# パラメータ
step = 50
lam  = 0.02   # TV正則化の強さ（大きいほど平滑化が強い）

# 画像とマスクの読み込み
img  = skimage.io.imread('data/MRI05.pgm').astype(float) / 255.0
mask = skimage.io.imread('data/mask_1d_rand40.tiff').astype(bool)
n    = img.shape[0]
if mask.shape != img.shape:
    mask = mask[:n, :n]

# k空間の部分サンプリング
Kfull = np.fft.fftshift(np.fft.fft2(np.fft.fftshift(img)))
Kobs  = Kfull * mask
print(f'サンプリング率: {mask.mean()*100:.0f}%')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img, cmap='gray')
axes[0].set_title('Original MRI')
axes[0].axis('off')
axes[1].imshow(mask, cmap='gray')
axes[1].set_title('Sampling mask')
axes[1].axis('off')
plt.show()

In [ ]:
# TV-CS 再構成ループ
sk = np.zeros_like(img)
psnr_history = []

for s in range(step):
    # データ整合性（観測k空間値で置換）
    Krec = np.fft.fftshift(np.fft.fft2(np.fft.fftshift(sk)))
    Krec[mask] = Kobs[mask]
    sk = np.real(np.fft.fftshift(np.fft.ifft2(np.fft.fftshift(Krec))))

    # TV収縮（Chambolle法）
    sk = skimage.restoration.denoise_tv_chambolle(sk, weight=lam, max_num_iter=10)

    psnr_history.append(skimage.metrics.peak_signal_noise_ratio(img, sk, data_range=1.0))

plt.plot(psnr_history)
plt.xlabel('Iteration')
plt.ylabel('PSNR [dB]')
plt.title('TV-CS: PSNR vs Iteration')
plt.grid(True)
plt.show()

In [ ]:
psnr_final = skimage.metrics.peak_signal_noise_ratio(img, sk, data_range=1.0)
ssim_final = skimage.metrics.structural_similarity(img, sk, data_range=1.0)

sk_zf = np.real(np.fft.fftshift(np.fft.ifft2(np.fft.fftshift(Kobs))))
psnr_zf = skimage.metrics.peak_signal_noise_ratio(img, sk_zf, data_range=1.0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(np.clip(sk_zf, 0, 1), cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'Zero-fill (PSNR={psnr_zf:.1f}dB)')
axes[1].axis('off')
axes[2].imshow(np.clip(sk, 0, 1), cmap='gray', vmin=0, vmax=1)
axes[2].set_title(f'TV-CS (PSNR={psnr_final:.1f}dB, SSIM={ssim_final:.3f})')
axes[2].axis('off')
fig.tight_layout()
fig.savefig('cs_tv_result.png', dpi=150)
plt.show()